In [ ]:
from pathlib import Path

TRAIN_CONFIG_PATH = Path(
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/configs/Training/train_7T.yaml"
)

SIMULATION_CONFIG_PATH = Path(
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/walinet/configs/Simulation/7T_on_the_fly.yaml"
)

BASIS_H5_PATH = Path(
    "/ceph/mri.meduniwien.ac.at/departments/radiology/"
    "mrsbrain/public/hfish/walinet/MetabModes/LCModelBasis/"
    "processed/walinet_7T_native_basis_v1.h5"
)

DEVICE = "cuda:0"
SEED = 123456

N_EXAMPLES = 100_000
BATCH_SIZE = 4096

In [ ]:
from pathlib import Path
import sys


project_root = Path.cwd()

while not (
    project_root
    / "src"
    / "walinet"
).is_dir():
    if project_root.parent == project_root:
        raise FileNotFoundError(
            "WALINET-Projektordner nicht gefunden."
        )

    project_root = project_root.parent


sys.path.insert(
    0,
    str(project_root / "src"),
)

print(
    "WALINET source:",
    project_root / "src",
)

from walinet.visualization.VisualizeSim import (
    plot_spectra_range,
)

In [ ]:
import torch
import yaml

from walinet.config.build import build_config
from walinet.config.build_simulation import (
    build_simulation_config,
)
from walinet.training_data.lcmodel_basis.acquisition import (
    prepare_basis_for_acquisition,
)
from walinet.training_data.metabolite_simulation import (
    MetaboliteSimulator,
)
from walinet.training_data.simulation_resources import (
    build_simulation_resources,
)
from walinet.training_data.spectrum_simulator import (
    SpectrumSimulator,
)


def load_yaml(path):
    with open(path, "r", encoding="utf-8") as file:
        return yaml.safe_load(file)


device = torch.device(DEVICE)

train_cfg = build_config(
    load_yaml(TRAIN_CONFIG_PATH),
    config_dir=TRAIN_CONFIG_PATH.parent,
)

simulation_cfg = build_simulation_config(
    load_yaml(SIMULATION_CONFIG_PATH),
    config_dir=SIMULATION_CONFIG_PATH.parent,
)

resources = build_simulation_resources(
    train_cfg=train_cfg,
    simulation_cfg=simulation_cfg,
)

pool = resources.train.to(device)

prepared_basis = prepare_basis_for_acquisition(
    BASIS_H5_PATH,
    target_bandwidth=(
        simulation_cfg.acquisition.bandwidth_hz
    ),
    target_n_timepoints=(
        simulation_cfg.acquisition.n_timepoints
    ),
)

metabolite_simulator = MetaboliteSimulator(
    prepared_basis=prepared_basis,
    config=simulation_cfg,
    device=device,
)

spectrum_simulator = SpectrumSimulator(
    pool=pool,
    metabolite_simulator=metabolite_simulator,
    config=simulation_cfg,
    max_retries=3,
)

generator = torch.Generator(
    device=device
)

generator.manual_seed(SEED)

In [ ]:
batch = spectrum_simulator.simulate(
    batch_size=N_EXAMPLES,
    generator=generator,
)

total_spectra = batch.normalized_input_spectra
baseline_spectra = batch.normalized_target_spectra

metabolite_spectra = (
    total_spectra
    - baseline_spectra
)

In [ ]:
plot_spectra_range(
    total_spectra=total_spectra,
    metabolite_spectra=metabolite_spectra,
    simulation_cfg=simulation_cfg,
    start=0,
    stop=5,
    component="real",
    ppm_min=0.0,
    ppm_max=7.0,
)